# Explicabilidad SHAP de Random Forest y red de atención

**Objetivo:** SHAP de RF y MLP; reentrena modelos, 35 épocas para la red.

Este cuaderno contiene el código completo del experimento `generate_shap_analysis.py`,
dividido en etapas y acompañado de explicaciones. No necesita ZIP, clonar un
repositorio ni ejecutar un script oculto. Ejecuta las celdas en orden.

**Recorrido:** entorno → carga e inspección → preparación → modelo o análisis →
evaluación → gráficos → descarga y colaboración.

**Entradas:** `dataset_entrenamiento_sierra_realista.csv`.

Los datos no se incluyen. Se conservan los valores predeterminados del protocolo histórico.
Las salidas se generarán al ejecutar; las figuras históricas no están incrustadas.


**Interpretación:** los pesos de atención, la importancia Gini y SHAP son indicadores distintos. No equivalen a causas físicas. El módulo de atención se conserva para documentar el experimento; su revisión debe evaluarse como una variante nueva.

## 1. Entorno y parámetros
Modifica los parámetros aquí para que una colaboración sea fácil de revisar. Los valores originales siguen seleccionados. Cada ejecución crea una carpeta nueva.

In [ ]:
# Instala únicamente las bibliotecas que falten en el entorno.
import importlib.util
import importlib.metadata
import subprocess
import sys
import os
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display, Image

DEPENDENCIAS = {'numpy': 'numpy', 'pandas': 'pandas', 'sklearn': 'scikit-learn', 'matplotlib': 'matplotlib', 'torch': 'torch', 'shap': 'shap'}
faltantes = [paquete for modulo, paquete in DEPENDENCIAS.items()
             if importlib.util.find_spec(modulo) is None]
if faltantes:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *faltantes], check=True)

CONFIG = dict(seed=42, rf_trees=100, rf_depth=15, epochs=35,
              learning_rate=0.002, batch_size=256)
EXPERIMENTO = 'shap'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
DATA_DIR = Path(globals().get('DATA_DIR_OVERRIDE', 'data')).resolve()
OUTPUT_DIR = Path(globals().get('OUTPUT_DIR_OVERRIDE', Path('results') / EXPERIMENTO / RUN_ID)).resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLBACKEND', 'Agg')
print('Parámetros:', CONFIG)
print('Salidas:', OUTPUT_DIR)

def mostrar_figura(ruta):
    # Conserva la figura en disco y también la muestra en el cuaderno.
    if str(ruta).lower().endswith('.png'):
        display(Image(filename=str(ruta)))


## 2. Carga e inspección de los datos
En Colab, la siguiente celda ofrece el selector si faltan archivos. Carga solo los CSV indicados, conservando sus nombres. En un entorno local, colócalos en `data/`. Para el dataset realista usa la copia de la subcarpeta `resultados` original.

In [ ]:
import pandas as pd

ARCHIVOS_REQUERIDOS = ['dataset_entrenamiento_sierra_realista.csv']
def localizar(nombre):
    candidatos = [DATA_DIR / nombre, DATA_DIR / 'resultados' / nombre]
    return next((p for p in candidatos if p.is_file()), candidatos[0])

pendientes = [n for n in ARCHIVOS_REQUERIDOS if not localizar(n).is_file()]
if pendientes:
    print('Archivos requeridos:', pendientes)
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError('Coloca los CSV en ' + str(DATA_DIR))
    cargados = files.upload()
    for nombre, contenido in cargados.items():
        if Path(nombre).name != nombre or nombre not in ARCHIVOS_REQUERIDOS:
            raise ValueError('Archivo inesperado: ' + nombre)
        (DATA_DIR / nombre).write_bytes(contenido)
    del cargados

DATA_FILES = {n: localizar(n) for n in ARCHIVOS_REQUERIDOS}
if any(not p.is_file() for p in DATA_FILES.values()):
    raise FileNotFoundError('Faltan CSV; repite esta celda y carga los indicados.')

# Inspección visible: dimensiones, tipos, valores faltantes y balance de clases.
for nombre, ruta in DATA_FILES.items():
    tabla = pd.read_csv(ruta)
    print('\n', nombre, '| filas y columnas:', tabla.shape)
    display(tabla.head())
    display(pd.DataFrame({'tipo': tabla.dtypes.astype(str), 'nulos': tabla.isna().sum()}))
    if 'clase' in tabla.columns:
        display(tabla['clase'].value_counts(dropna=False).rename('filas').to_frame())

REGISTRO = dict(experimento=EXPERIMENTO, parametros=CONFIG, run_id=RUN_ID,
    input_sha256={n: hashlib.sha256(p.read_bytes()).hexdigest() for n, p in DATA_FILES.items()},
    versiones={p: importlib.metadata.version(p) for p in DEPENDENCIAS.values()},
    estado='iniciado')
(OUTPUT_DIR / 'ejecucion.json').write_text(json.dumps(REGISTRO, ensure_ascii=False, indent=2), encoding='utf-8')


In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import shap
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
BASE_DIR = str(OUTPUT_DIR)
FILE_DATASET = str(DATA_FILES['dataset_entrenamiento_sierra_realista.csv'])
FILE_PNG_SHAP_RF = str(OUTPUT_DIR / 'shap_summary_rf.png')
FILE_PNG_SHAP_NN = str(OUTPUT_DIR / 'shap_summary_attention.png')
FILE_CSV_SHAP = str(OUTPUT_DIR / 'shap_importances_realista.csv')
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])


### Definición: `FeatureSelfAttention`
Esta definición queda visible para revisar su arquitectura y proponer variantes. Sus valores originales se conservan.

In [ ]:
class FeatureSelfAttention(nn.Module):
    def __init__(self, input_dim):
        super(FeatureSelfAttention, self).__init__()
        self.query = nn.Linear(input_dim, input_dim, bias=False)
        self.key = nn.Linear(input_dim, input_dim, bias=False)
        self.value = nn.Linear(input_dim, input_dim, bias=False)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)
        attention_scores = torch.matmul(q.unsqueeze(2), k.unsqueeze(1)) / np.sqrt(x.shape[1])
        attention_weights = self.softmax(attention_scores.mean(dim=2))
        attended_x = v * attention_weights
        return attended_x


### Definición: `FireAttentionMLP`
Esta definición queda visible para revisar su arquitectura y proponer variantes. Sus valores originales se conservan.

In [ ]:
class FireAttentionMLP(nn.Module):
    def __init__(self, input_dim):
        super(FireAttentionMLP, self).__init__()
        self.attention = FeatureSelfAttention(input_dim)
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        attended_x = self.attention(x)
        output = self.classifier(attended_x)
        return output


## 4. Desarrollo del experimento
Las siguientes celdas contienen el cuerpo del análisis original, con comentarios y código editable. Puedes inspeccionar las variables entre pasos. Ejecuta nuevamente desde la carga de datos si cambias una transformación previa.

### Etapa 1: Carga y preparación de datos
Comprueba las columnas, las filas descartadas y el significado de la etiqueta antes de entrenar. El código conserva la selección de variables del experimento original.

In [ ]:
print("="*75)
print("  GENERANDO ANÁLISIS DE EXPLICABILIDAD SHAP PARA EL PAPER")
print("="*75)

if not os.path.exists(FILE_DATASET):
    print(f"[ERROR] No se encuentra el dataset: {FILE_DATASET}")
    raise ValueError("Revisa el archivo y las columnas indicadas en la salida anterior.")
df = pd.read_csv(FILE_DATASET).dropna()
features = ['elevacion', 'pendiente', 'orientacion', 'ndvi', 'evi', 'nbr', 'temp_c', 'precip_mm']

nombres_es = {
    'temp_c': 'Temperatura',
    'precip_mm': 'Precipitación',
    'elevacion': 'Elevación',
    'ndvi': 'NDVI (Verdor)',
    'nbr': 'NBR (Humedad)',
    'evi': 'EVI (Biomasa)',
    'pendiente': 'Pendiente',
    'orientacion': 'Orientación'
}

df_renamed = df[features].rename(columns=nombres_es)
feature_names_es = list(df_renamed.columns)

X = df[features].values
y = df['clase'].values


### Etapa 2: Atribuciones SHAP
Inspecciona la muestra explicada, el conjunto de referencia y la clase de salida. SHAP describe el comportamiento del modelo; no demuestra causalidad. KernelExplainer puede tardar varios minutos.

In [ ]:

# Dividir datos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=CONFIG['seed'], stratify=y)

# Submuestra para acelerar el cálculo de SHAP (1000 muestras de prueba)
test_idx = np.random.choice(len(X_test), 1000, replace=False)
X_test_sample = X_test[test_idx]
df_test_sample = pd.DataFrame(X_test_sample, columns=feature_names_es)

# --- 1. SHAP PARA RANDOM FOREST ---
print("\n[1/2] Calculando valores SHAP para Random Forest (TreeExplainer)...")
rf = RandomForestClassifier(n_estimators=CONFIG['rf_trees'], max_depth=CONFIG['rf_depth'], random_state=CONFIG['seed'], n_jobs=-1)
rf.fit(X_train, y_train)

explainer_rf = shap.TreeExplainer(rf)
shap_values_rf = explainer_rf.shap_values(X_test_sample)

# Para clasificación binaria, tomamos la clase 1 (Incendio)
if isinstance(shap_values_rf, list):
    shap_vals_rf = shap_values_rf[1]
else:
    shap_vals_rf = shap_values_rf[:, :, 1] if len(shap_values_rf.shape) == 3 else shap_values_rf

# Graficar y guardar SHAP Beeswarm Plot para RF
plt.figure(figsize=(10, 6), dpi=300)
shap.summary_plot(shap_vals_rf, df_test_sample, show=False)


### Etapa 3: Entrenamiento de la red
Se optimizan los pesos con los datos de entrenamiento. El registro muestra la pérdida de la última tanda; no es una métrica de validación. Mantén fijo el conjunto de prueba al comparar configuraciones.

In [ ]:
plt.title("Explicabilidad SHAP: Random Forest (Clase Incendio)", fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(FILE_PNG_SHAP_RF, bbox_inches='tight')
mostrar_figura(FILE_PNG_SHAP_RF)
plt.close()
print(f"[OK] Gráfico SHAP Random Forest guardado en: {FILE_PNG_SHAP_RF}")

# --- 2. SHAP PARA RED NEURONAL DE AUTOTENSIÓN ---
print("\n[2/2] Calculando valores SHAP para Red de Autotensión (KernelExplainer)...")
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
X_test_sample_s = X_test_s[test_idx]

# Entrenar modelo rápido PyTorch
model = FireAttentionMLP(input_dim=len(features))
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=1e-5)

ds = TensorDataset(torch.FloatTensor(X_train_s), torch.FloatTensor(y_train).unsqueeze(1))
ld = DataLoader(ds, batch_size=CONFIG['batch_size'], shuffle=True)

model.train()
for epoca in range(CONFIG['epochs']):
    for bx, by in ld:
        optimizer.zero_grad()
        out = model(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
    if epoca == 0 or (epoca + 1) % 10 == 0 or epoca + 1 == CONFIG['epochs']:
        print(f"Época {epoca + 1}/{CONFIG['epochs']} | pérdida última tanda: {float(loss.detach()):.5f}")


### Etapa 4: Atribuciones SHAP
Inspecciona la muestra explicada, el conjunto de referencia y la clase de salida. SHAP describe el comportamiento del modelo; no demuestra causalidad. KernelExplainer puede tardar varios minutos.

In [ ]:

model.eval()

# Función wrapper para KernelExplainer
def predict_nn(x_numpy):
    x_tensor = torch.FloatTensor(x_numpy)
    with torch.no_grad():
        out = model(x_tensor).numpy()
    return out.flatten()

# Usar una submuestra de background (100 puntos) para acelerar KernelExplainer
background = X_train_s[np.random.choice(len(X_train_s), 100, replace=False)]
explainer_nn = shap.KernelExplainer(predict_nn, background)

# Calcular SHAP en 300 puntos para rendimiento óptimo
shap_sample_idx = np.random.choice(len(X_test_sample_s), 300, replace=False)
shap_vals_nn = explainer_nn.shap_values(X_test_sample_s[shap_sample_idx])

if isinstance(shap_vals_nn, list):
    shap_vals_nn = shap_vals_nn[0]

if np.ndim(shap_vals_nn) == 3:
    shap_vals_nn = np.asarray(shap_vals_nn)[:, :, 0]

# Graficar y guardar SHAP Beeswarm Plot para Red de Autotensión
plt.figure(figsize=(10, 6), dpi=300)


### Etapa 5: Atribuciones SHAP
Inspecciona la muestra explicada, el conjunto de referencia y la clase de salida. SHAP describe el comportamiento del modelo; no demuestra causalidad. KernelExplainer puede tardar varios minutos.

In [ ]:
shap.summary_plot(shap_vals_nn, df_test_sample.iloc[shap_sample_idx], show=False)
plt.title("Explicabilidad SHAP: Red Neuronal de Autotensión (Clase Incendio)", fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(FILE_PNG_SHAP_NN, bbox_inches='tight')
mostrar_figura(FILE_PNG_SHAP_NN)
plt.close()
print(f"[OK] Gráfico SHAP Red de Autotensión guardado en: {FILE_PNG_SHAP_NN}")

# Guardar tabla con importancias absolutas promedio de SHAP (mean |SHAP value|)
mean_abs_rf = np.abs(shap_vals_rf).mean(axis=0).flatten()
mean_abs_nn = np.abs(shap_vals_nn).mean(axis=0).flatten()

df_shap_summary = pd.DataFrame({
    'Variable': feature_names_es,
    'Mean |SHAP| (Random Forest)': mean_abs_rf,
    'Mean |SHAP| (Attention Net)': mean_abs_nn
}).sort_values(by='Mean |SHAP| (Attention Net)', ascending=False).reset_index(drop=True)

df_shap_summary.to_csv(FILE_CSV_SHAP, index=False)
print("\n" + "="*65)
print("  TABLA RESUMEN DE VALORES SHAP PROMEDIO (MEAN |SHAP VALUE|)")
print("="*65)
print(df_shap_summary.round(4).to_string(index=False))
print("="*75)


## 5. Inspeccionar y guardar resultados
Las tablas se muestran debajo. El registro identifica parámetros, versiones y hashes. Esta celda marca el registro como completado: ejecútala solo cuando hayan terminado todas las etapas anteriores.

In [ ]:
for ruta in sorted(OUTPUT_DIR.glob('*.csv')):
    print(ruta.name)
    display(pd.read_csv(ruta).head(20))
REGISTRO['estado'] = 'completado'
(OUTPUT_DIR / 'ejecucion.json').write_text(json.dumps(REGISTRO, ensure_ascii=False, indent=2), encoding='utf-8')
print('Salidas:', [p.name for p in sorted(OUTPUT_DIR.iterdir())])


In [ ]:
# Descarga las salidas antes de cerrar una sesión de Colab.
import shutil
paquete = shutil.make_archive(str(OUTPUT_DIR.parent / (EXPERIMENTO + '-' + RUN_ID)), 'zip', OUTPUT_DIR)
try:
    from google.colab import files
except ImportError:
    print('ZIP guardado en:', paquete)
else:
    files.download(paquete)


## 6. Proponer una colaboración

1. Plantea una pregunta concreta: por ejemplo, el efecto de un hiperparámetro o de una variable.
2. Conserva una ejecución con la configuración original y luego cambia una sola decisión.
3. Registra el hash del dataset, la partición, la semilla y todas las métricas relevantes.
4. En GitHub, abre un Issue para discutir la propuesta. Trabaja en un fork o en una rama
   con permisos y envía un pull request que explique el cambio y su evidencia.
5. Antes del commit, borra las salidas del cuaderno y revisa que no incluyan datos privados.
   Adjunta resultados únicamente cuando tengas permiso para compartirlos.

Edita la siguiente celda Markdown para documentar tu propuesta. No se envía nada automáticamente.

### Registro de la propuesta

- **Autor y objetivo:**
- **Hipótesis:**
- **Parámetro o código modificado:**
- **Dataset y hash:**
- **Semilla y partición:**
- **Resultado original y resultado de la variante:**
- **Limitaciones y conclusión:**
